In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/miketattershall/lora-adapter/lora-adapter/adapter_model.safetensors
/kaggle/input/datasets/miketattershall/lora-adapter/lora-adapter/adapter_config.json
/kaggle/input/datasets/miketattershall/lora-adapter/lora-adapter/README.md
/kaggle/input/datasets/miketattershall/lora-adapter/lora-adapter/tokenizer.json
/kaggle/input/datasets/miketattershall/lora-adapter/lora-adapter/tokenizer_config.json
/kaggle/input/datasets/miketattershall/lora-adapter/lora-adapter/chat_template.jinja


In [2]:
# CELL 1 — install Unsloth + Ollama
!pip install -q unsloth
!apt-get -qq install -y zstd 2>/dev/null; curl -fsSL https://ollama.com/install.sh | sh
print("✅ installs done")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 865.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 94.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 106.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/

In [3]:
# CELL 2 (fixed) — convert in /tmp (1.1 TB free), save final GGUF to /kaggle/working
import os, glob, zipfile, subprocess, time, requests, shutil

# 1) locate adapter (extract to /tmp if zipped)
cfg = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
if not cfg:
    z = glob.glob("/kaggle/input/**/*.zip", recursive=True)[0]
    print("extracting:", z)
    with zipfile.ZipFile(z) as zf:
        for info in zf.infolist():
            n = info.filename.replace("\\", "/")
            if n.endswith("/") or not n.strip():
                continue
            t = "/tmp/adapter_raw/" + n
            os.makedirs(os.path.dirname(t), exist_ok=True)
            with zf.open(info) as s, open(t, "wb") as d:
                d.write(s.read())
    cfg = glob.glob("/tmp/adapter_raw/**/adapter_config.json", recursive=True)
ADAPTER = os.path.dirname(cfg[0])
print("ADAPTER:", ADAPTER, "->", os.listdir(ADAPTER))

# 2) export to GGUF in /tmp (tons of space)
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=ADAPTER, max_seq_length=4096, dtype=None, load_in_4bit=True)
model.save_pretrained_gguf("/tmp/mycoder-tuned", tokenizer, quantization_method="q4_k_m")

# 3) copy ONLY the final Q4 gguf + Modelfile into /kaggle/working (to save)
ggufs = glob.glob("/tmp/**/*.gguf", recursive=True)
gguf = next((g for g in ggufs if "Q4" in g.upper()), ggufs[0])
gdir = os.path.dirname(gguf)
os.makedirs("/kaggle/working/mycoder-tuned", exist_ok=True)
shutil.copy(gguf, "/kaggle/working/mycoder-tuned/")
shutil.copy(f"{gdir}/Modelfile", "/kaggle/working/mycoder-tuned/")
print("✅ saved to /kaggle/working/mycoder-tuned:", os.listdir("/kaggle/working/mycoder-tuned"))

# 4) quick register/test with ollama
os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
subprocess.Popen(["ollama", "serve"], stdout=open("/tmp/ol.log", "w"), stderr=subprocess.STDOUT)
time.sleep(6)
os.system(f"ollama create mycoder-tuned -f {gdir}/Modelfile")
print("models:", [m['name'] for m in requests.get('http://localhost:11434/api/tags').json()['models']])

ADAPTER: /kaggle/input/datasets/miketattershall/lora-adapter/lora-adapter -> ['adapter_model.safetensors', 'adapter_config.json', 'README.md', 'tokenizer.json', 'tokenizer_config.json', 'chat_template.jinja']
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


/usr/local/lib/python3.12/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['lora_ga_config', 'use_bdlora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


==((====))==  Unsloth 2026.6.7: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

...


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [01:48<00:00, 27.19s/it]


Unsloth: Merge process complete. Saved to `/tmp/mycoder-tuned`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b9637 (llama-b9637-bin-ubuntu-x64.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/tmp/mycoder-tuned_gguf/qwen2.5-coder-7b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Genera

gathering model components ⠋ gathering model components ⠙ gathering model components ⠹ gathering model components ⠸ gathering model components ⠼ gathering model components ⠦ gathering model components ⠦ gathering model components ⠧ gathering model components ⠏ gathering model components ⠋ gathering model components ⠋ gathering model components ⠹ gathering model components ⠸ gathering model components ⠸ gathering model components ⠴ gathering model components ⠴ gathering model components ⠦ gathering model components ⠇ gathering model components ⠇ gathering model components ⠏ gathering model components ⠋ gathering model components ⠙ gathering model components ⠹ gathering model components ⠸ gathering model components ⠼ gathering model components ⠴ gathering model components ⠦ gathering model components ⠧ gathering model components ⠏ gathering model components ⠏ gathering model components ⠋ gathering model components ⠹ gathering model components ⠹ gathering model components ⠸ gathering mode

models: ['mycoder-tuned:latest']


gathering model components 
copying file sha256:f747784e6b9c7afdfadcae78a07c45cc2408c7756b05f771e81853987dd3862f 100% 
parsing GGUF 
verifying conversion 
using existing layer sha256:f747784e6b9c7afdfadcae78a07c45cc2408c7756b05f771e81853987dd3862f 
creating new layer sha256:1e65450c30670713aa47fe23e8b9662bdf4065e81cc8e3cbfaa98924fcc0d320 
creating new layer sha256:66b9ea09bd5b7099cbb4fc820f31b575c0366fa439b08245566692c6784e281e 
writing manifest 
success 


In [4]:
# clean the failed partial export + check where there's free space
import shutil, glob
for p in glob.glob("/kaggle/working/mycoder-tuned*") + glob.glob("/kaggle/working/adapter_raw"):
    shutil.rmtree(p, ignore_errors=True)
print("cleaned partial output\n")

for p in ["/kaggle/working", "/kaggle/temp", "/tmp", "/root", "/"]:
    try:
        t, u, f = shutil.disk_usage(p)
        print(f"{p:16} free {f/1e9:5.1f} GB / total {t/1e9:5.1f} GB")
    except Exception as e:
        print(f"{p:16} n/a ({e})")

cleaned partial output

/kaggle/working  free  20.9 GB / total  21.0 GB
/kaggle/temp     n/a ([Errno 2] No such file or directory: '/kaggle/temp')
/tmp             free 1150.0 GB / total 8656.9 GB
/root            free 1150.0 GB / total 8656.9 GB
/                free 1150.0 GB / total 8656.9 GB
